# DATALUS POC Quickstart: SIH-SUS Generative Pipeline

This notebook demonstrates the **DATALUS** framework for synthetic tabular data generation. 
We use a real-world sample of Brazilian health data (SIH-SUS) to showcase the entire pipeline: 
from raw data ingestion to ONNX edge export, including auditing and generative operations.

**DATALUS** is designed for high-dimensional, heterogeneous government datasets, ensuring privacy (LGPD alignment) 
and utility through advanced diffusion models.

## 1. Environment Setup
Detecting environment and installing dependencies if necessary.

In [ ]:
import os
import sys
from pathlib import Path

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_poc'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_poc'
    return './datalus_poc'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

INSTALLED = Path(WORKING_DIR) / '.installed'

if not INSTALLED.exists():
    if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        !git clone --depth 1 https://github.com/emanuellcs/datalus.git "{WORKING_DIR}/source" 2>/dev/null || true
        %cd "{WORKING_DIR}/source"
        !pip install --quiet -e '.[dev]' pysus polars matplotlib nest_asyncio 2>&1 | tail -5
        sys.path.insert(0, f'{WORKING_DIR}/source')
        INSTALLED.touch()
        print()
        print('=' * 60)
        print('  INSTALLATION COMPLETE')
        print('  Click "RESTART RUNTIME" in the warning above,')
        print('  or go to Runtime > Restart and run all')
        print('=' * 60)
        os._exit(0)
    else:
        INSTALLED.touch()
        print('Local environment. Dependencies assumed installed.')
else:
    src = Path(WORKING_DIR) / 'source'
    if src.is_dir():
        sys.path.insert(0, str(src))
    print('Dependencies already installed.')


## 2. Data Ingestion
We fetch Brazilian public health data through multiple fallback strategies.
The primary source is SIH-SUS clinical records (group `RD`) containing
the `MORTE` column (death indicator). If SIH is unavailable, we try
SIM (mortality), SINAN (disease notifications), direct DATASUS FTP,
or a built-in sample to ensure the pipeline always has data to process.
This dataset-agnostic approach works with any Brazilian government sector.

In [ ]:
from pysus import sih, sim, sinan
import polars as pl
import numpy as np
from datetime import datetime
from pathlib import Path
import urllib.request
import asyncio

print('Downloading clinical data from DATASUS...')

STATES = ['SP', 'RJ', 'MG', 'PR', 'RS', 'BA', 'DF']

def fmt_tier(label, state, year, month, count):
    return f'Tier {label}: {state}/{year}/{month} ({count} records)'

def try_pysus():
    now = datetime.now()
    sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
    for state in STATES:
        for y in range(sy, sy - 2, -1):
            for m in range(sm if y == sy else 12, 0, -1):
                try:
                    df = sih(state=state, year=y, month=[m], group='RD', as_dataframe=True)
                    if df is not None and len(df) > 0 and 'MORTE' in df.columns:
                        print(fmt_tier('1: SIH-RD', state, y, m, len(df)))
                        return pl.from_pandas(df)
                except Exception:
                    continue
        for y in range(sy, sy - 2, -1):
            try:
                df = sim(state=state, year=y, as_dataframe=True)
                if df is not None and len(df) > 0:
                    print(fmt_tier('2: SIM', state, y, 'all', len(df)))
                    df['MORTE'] = 1
                    return pl.from_pandas(df)
            except Exception:
                continue
    for y in range(sy, sy - 2, -1):
        try:
            df = sinan(disease='deng', year=y, as_dataframe=True)
            if df is not None and len(df) > 0:
                print(fmt_tier('3: SINAN-deng', 'BR', y, 'all', len(df)))
                if 'MORTE' not in df.columns:
                    df['MORTE'] = 0
                return pl.from_pandas(df)
        except Exception:
            continue
    return None

def try_direct_ftp():
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        return None
    from pysus.api.extensions import DBC
    now = datetime.now()
    sy, sm = (now.year, now.month - 3) if now.month > 3 else (now.year - 1, now.month + 9)
    base = 'ftp://ftp.datasus.gov.br/dissemin/publicos/SIHSUS/199301_/Dados/'
    for state in ['SP', 'RJ', 'MG']:
        for y in range(sy, sy - 2, -1):
            for m in range(sm if y == sy else 12, 0, -1):
                fname = f'RD{state}{y % 100:02d}{m:02d}.dbc'
                try:
                    with urllib.request.urlopen(base + fname, timeout=30) as r:
                        dbc_path = Path('/tmp') / fname
                        dbc_path.write_bytes(r.read())
                    df = asyncio.run(DBC(path=dbc_path).load())
                    if df is not None and len(df) > 0:
                        print(fmt_tier('4: FTP direct', state, y, m, len(df)))
                        return pl.from_pandas(df)
                except Exception:
                    continue
    return None

def generate_sample():
    print('Tier 5: Generated demo sample (real data unavailable)')
    print('NOTE: This is not real data — results are for pipeline demo only.')
    np.random.seed(42)
    n = 1000
    return pl.DataFrame({
        'MORTE': np.random.choice([0, 1], n, p=[0.92, 0.08]).astype(np.int64),
        'IDADE': np.random.randint(0, 100, n).astype(np.float64),
        'SEXO': np.random.choice(['0', '1'], n),
        'DIAS_PERMANENCIA': np.clip(np.random.poisson(5, n), 0, 100).astype(np.float64),
        'VALOR_TOTAL': np.round(np.random.lognormal(7, 1.5, n), 2),
        'PROCEDIMENTO_PRINCIPAL': np.random.choice(['AB', 'CD', 'EF', 'GH', 'IJ'], n),
        'MUNICIPIO_MOV': np.random.choice(['3550308', '3304557', '3106200'], n),
    })

df_polars = try_pysus()
if df_polars is None:
    df_polars = try_direct_ftp()
if df_polars is None:
    df_polars = generate_sample()

df_polars = df_polars.with_columns(
    pl.col('MORTE').cast(pl.Int64).fill_null(0)
)

cols_to_keep = [
    c for c in df_polars.columns
    if not c.startswith(('N_AIH', 'IDENT', 'CNS', 'CPF', 'CNPJ'))
]
df_polars = df_polars.select(
    ['MORTE'] + [c for c in cols_to_keep if c != 'MORTE']
).head(10000)

raw_path = f'{WORKING_DIR}/raw_sih.parquet'
df_polars.write_parquet(raw_path)
print(f'Data saved to {raw_path} | Shape: {df_polars.shape}')
print(f'Columns ({len(df_polars.columns)}): {df_polars.columns[:8]}...')

## 3. Ingesting with DATALUS CLI
The `ingest` command infers the schema, detects delimiters, and prepares the data for modeling.

In [ ]:
!datalus ingest {WORKING_DIR}/raw_sih.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE

## 4. Training the Diffusion Model
We train the residual MLP denoiser. For this POC, we use a very low number of epochs.

In [ ]:
!datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 3 --batch-size 1024 --gpu 0

## 5. Ab-initio Generation (Sampling)
Generating entirely new synthetic records from the learned distribution.

In [ ]:
!datalus sample {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet --n-records 5000

## 6. Data Augmentation
Appending synthetic rows to a small existing dataset.

In [ ]:
synthetic = pl.read_parquet(f'{WORKING_DIR}/synthetic.parquet')
seed_for_augment = synthetic.sample(n=1000)
seed_for_augment.write_parquet(f'{WORKING_DIR}/seed_for_augment.parquet')

!datalus augment {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/seed_for_augment.parquet {WORKING_DIR}/augmented.parquet --n-records 1000

## 7. Minority Class Balancing
Generating records to balance specific target classes (e.g., mortality).

In [ ]:
!datalus balance {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/balanced.parquet MORTE '{"1": 2000}'

## 8. Tabular Inpainting (RePaint)
Filling missing values (nulls) while preserving observed fields.

In [ ]:
processed = pl.read_parquet(f'{WORKING_DIR}/processed.parquet')
# Create missing values in 'IDADE' column for demonstration
mask = np.random.rand(100) > 0.5
incomplete = processed.head(100).with_columns(
    pl.when(pl.lit(mask)).then(None).otherwise(pl.col('IDADE')).alias('IDADE')
)
incomplete.write_parquet(f'{WORKING_DIR}/incomplete.parquet')

!datalus inpaint {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/incomplete.parquet {WORKING_DIR}/inpainted.parquet

## 9. Counterfactual Modification
Applying interventions (e.g., changing 'SEXO') and regenerating compatible fields.

In [ ]:
!datalus counterfactual {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/counterfactual.parquet '{"SEXO": "1"}'

## 10. Autonomous Privacy & Utility Audit
Evaluating DCR metrics and TSTR/TRTR utility ratios.

In [ ]:
!datalus audit {WORKING_DIR}/processed.parquet {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/schema_config.json {WORKING_DIR}/audit_report.json --target-column MORTE --mia-mode ci_lite

## 11. Visualizing Audit Results


In [ ]:
import json
import matplotlib.pyplot as plt

with open(f'{WORKING_DIR}/audit_report.json', 'r') as f:
    report = json.load(f)

print(json.dumps(report, indent=2))

# Plot Utility (TSTR vs TRTR)
if 'utility' in report and 'target_class_probs' in report['utility']:
    probs = report['utility']['target_class_probs']
    labels = list(probs.keys())
    real_probs = [probs[l]['real'] for l in labels]
    synth_probs = [probs[l]['synthetic'] for l in labels]
    
    x = np.arange(len(labels))
    width = 0.35
    
    fig, ax = plt.subplots()
    ax.bar(x - width/2, real_probs, width, label='Real')
    ax.bar(x + width/2, synth_probs, width, label='Synthetic')
    
    ax.set_ylabel('Probability')
    ax.set_title('Target Class Distribution Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.legend()
    plt.show()

## 12. Exporting to ONNX Edge
Exporting weights for browser-local or edge inference.

In [ ]:
!datalus export-onnx {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/artifacts --quantize

## 13. Serving and Web Deployment
Artifacts can be served via FastAPI:
`datalus serve {WORKING_DIR}/artifacts --host 0.0.0.0 --port 8000`

This enables the React WASM frontend to perform inference locally in the user's browser.